# step3 — 코드 신호 인과 (RQ2 확정)

**무엇을 확인하나:** 문맥의 **위반 이름**(예: `parse_header`)의 모델 속을 **준수판**으로 바꿔치기해서,
준수 선호가 **되돌아오는지**를 잰다. 세 방식으로 나눠 본다 — **어텐션만 / 내용만 / 둘다**.
**내용만 바꿔도 되돌아오고 어텐션만은 안 되면 → 코드 신호는 '얼마나 보나'가 아니라 '담긴 내용(Value)'.**
관측(step2)의 첫 단서를 여기서 **인과로 확정**한다.

**고정 설정:** 4모델 · 함수이름 504개(42묶음) · 무작위값 42 · 평균 덮어쓰기 · 문맥은 준수 6 + 위반 6.

**바꿔넣는 이름 3종**(위반 이름을 무엇의 속으로 덮나):
| 이름 | 예 | 역할 |
|---|---|---|
| 같은 이름 camel판 | `parse_header`→`parseHeader` | 주 실험 |
| 다른 camel 이름 | `sortBuffer` | 형태만 같음(형태 통제) |
| 다른 snake 이름 | `count_vowels` | 음성통제(안 되돌아와야 정상) |

> **부하 주의:** 전 층 × 3방식이라 무겁다. 조건 = 42묶음 × 이름 3종 = **126개/모델**, 각각 전 층 스윕.
> **deepseek-6.7b는 T4에서 메모리 초과(OOM) 가능** → 셀 ④에서 그 모델에 8bit를 켠다(주석 참고).

**모델 하나씩 돌린다.** 셀 ④ `PICK`에서 모델 1개 → 셀 ⑤~⑦. 끝나면 `PICK` 바꿔 반복. 끊겨도 이미 저장된 건 건너뛴다.


In [ ]:
# ① 환경 설정 — 설치, GPU 확인, 무작위값 42 고정
!pip install -q transformers accelerate torch matplotlib pandas bitsandbytes

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (매우 느림)')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('무작위값 고정:', SEED)


In [ ]:
# ② 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin step3/code-cause
!git checkout step3/code-cause
!git pull --quiet origin step3/code-cause
!pip install -e . -q
import sys; sys.path.insert(0, 'src')


In [ ]:
# ③ 조건 설정 — 4모델 중 하나 골라 42묶음 x 바꿔넣는이름 3종
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation, Intervention, InterventionKind)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',           family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-6.7b-instruct',  family='deepseek',  dtype='float16'),
    ModelSpec(name='unsloth/Llama-3.2-3B-Instruct',             family='llama',     dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',       family='stability', dtype='float16'),
]

# ★ 이번에 돌릴 모델 하나 (0=qwen, 1=deepseek, 2=llama, 3=stable)
PICK = 0
MODEL = MODELS[PICK]
# deepseek-6.7b OOM나면 아래 해제(8bit):
# if MODEL.family=='deepseek': MODEL = ModelSpec(name=MODEL.name, family='deepseek', dtype='float16', quantization='8bit')

# 바꿔넣는 이름 3종
SWAP_FROM = ['compliant', 'unrelated_camel', 'unrelated_snake']
BLOCKS = list(range(42))

def cond(block, swap):
    return Condition(model=MODEL,
        preceding=PrecedingCode(n_compliant=6, n_functions=12, composition=Composition.POOL, pool_block=block),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL),
        intervention=Intervention(kind=InterventionKind.KEY_VALUE, layers='sweep', donor=swap, target='code'),
        token_unit='mean', seed=42)

conditions = [cond(b, s) for s in SWAP_FROM for b in BLOCKS]
print('모델:', MODEL.family, '| 조건 수:', len(conditions), '(=바꿔넣는이름 3종 x 묶음 42)')
print('예:', conditions[0].slug())


In [ ]:
# ④ 실행 — 개입(전 층 스윕). 조건마다 즉시 저장(재개). KV 바꿔치기라 어텐션 안 꺼내도 됨(빠름).
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
import numpy as np

STEP = 'step3_code-cause'
todo = [c for c in conditions if not result_path(c, step=STEP).exists()]
print(f'[{MODEL.family}] 전체 {len(conditions)} / 남은 {len(todo)}')

if todo:
    handle = load_model(MODEL)   # 개입은 output_attentions 불필요
    print(f'  층수 {handle.num_layers} | GQA {handle.gqa_info()}')
    for i, c in enumerate(todo, 1):
        out = run(c, handle=handle)   # 개입(전 층 스윕) — 되돌림 정도 측정
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics, step=STEP, rq='RQ2'))
        if i % 5 == 0 or i == len(todo):
            pl = out.metrics.per_layer; ex = out.metrics.extra
            cand = [(int(L), v.get('value__recovery')) for L, v in pl.items() if v.get('value__recovery') is not None]
            gap = ex['S_clean'] - ex['S_base']
            if cand:
                bL, bV = max(cand, key=lambda t: t[1])
                print(f'    [{i}/{len(todo)}] 내용만 최고 되돌림 {bV:.2f} @L{bL}  (깨끗-위반 차 {gap:+.2f})')
            else:
                print(f'    [{i}/{len(todo)}] 되돌림 계산 불가 (깨끗-위반 차 {gap:+.2f})')
    print('  완료.')
else:
    print('  이미 다 됨')


In [ ]:
# ⑤ 결과 로드 (이 모델 것)
from harness import result_path
from harness.results import load_result
recs = [load_result(result_path(c, step='step3_code-cause')) for c in conditions
        if result_path(c, step='step3_code-cause').exists()]
print('불러온 조건:', len(recs), '-> results/step3_code-cause/')


In [ ]:
# ⑥ 요약 — 바꿔넣는 이름별로 '어텐션만/내용만/둘다' 되돌림 정도를 층별 평균 (판정불가 분리) + 그림
import numpy as np, pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt

SWAP_KO = {'compliant':'같은 이름 camel판', 'unrelated_camel':'다른 camel 이름', 'unrelated_snake':'다른 snake 이름(음성통제)'}
SWAP_EN = {'compliant':'Same name (camel)', 'unrelated_camel':'Other camel name', 'unrelated_snake':'Other snake (neg. ctrl)'}
WAY_KO  = {'key':'어텐션만', 'value':'내용만', 'key_value':'둘다'}
WAY_EN  = {'key':'Attention only', 'value':'Content only', 'key_value':'Both'}
GAP_MIN = 1.0   # 깨끗-위반 차가 이보다 작으면 '판정 불가'(되돌림 계산 무의미)

# 바꿔넣는이름 -> 방식 -> 층 -> [묶음별 되돌림]
agg = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
counts = defaultdict(lambda: {'valid':0, 'undecidable':0})
for r in recs:
    swap = r.condition.intervention.donor
    ex = r.metrics.extra; gap = ex['S_clean'] - ex['S_base']
    if abs(gap) < GAP_MIN:
        counts[swap]['undecidable'] += 1; continue
    counts[swap]['valid'] += 1
    for L, v in r.metrics.per_layer.items():
        L = int(L)
        for way in ['key','value','key_value']:
            rec = v.get(f'{way}__recovery')
            if rec is not None: agg[swap][way][L].append(rec)

for swap in SWAP_FROM:
    c = counts[swap]
    print(f'[{SWAP_KO[swap]}] 유효 묶음 {c["valid"]} / 판정불가 {c["undecidable"]}')

def curve(swap, way):
    d = agg[swap][way]; layers = sorted(d)
    return layers, [float(np.mean(d[L])) for L in layers]

# 같은 이름 camel판에서 내용만 최고 되돌림 층
layers, ys = curve('compliant','value')
if layers:
    bi = int(np.argmax(ys))
    print(f'\n[핵심] 같은 이름·내용만: 최고 되돌림 {ys[bi]:.2f} @L{layers[bi]} (전체 {len(layers)}층 중)')
    lk, yk = curve('compliant','key')
    if lk: print(f'       참고 같은 이름·어텐션만: 최고 되돌림 {max(yk):.2f}')

# 그림: 바꿔넣는이름 3종 x (어텐션만/내용만/둘다) — 라벨은 영문(글자 깨짐 방지)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for ax, swap in zip(axes, SWAP_FROM):
    for way, color in [('key','tab:gray'), ('value','tab:blue'), ('key_value','tab:orange')]:
        lx, yy = curve(swap, way)
        if lx: ax.plot(lx, yy, label=WAY_EN[way], color=color, lw=2)
    ax.axhline(0, color='k', lw=.6); ax.axhline(1, color='k', lw=.6, ls=':')
    ax.set_title(SWAP_EN[swap]); ax.set_xlabel('Layer'); ax.grid(alpha=.25); ax.legend(fontsize=8)
axes[0].set_ylabel('Recovery (0=none, 1=full)')
plt.suptitle(f'{MODEL.family} — step3 code causal (recovery by layer)')
plt.tight_layout(); plt.show()


In [ ]:
# ⑦ 결과 폴더 zip 다운로드
import shutil
shutil.make_archive('step3_code-cause_results', 'zip', 'results/step3_code-cause')
try:
    from google.colab import files; files.download('step3_code-cause_results.zip')
except Exception as e:
    print('Colab 아님(로컬):', e)
